In [ ]:
!pip install tensorflow

In [31]:
# importing libs
import pandas as pd
import numpy as np
import re
import pickle

import tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [24]:
# loading data
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])


In [25]:
#cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_message'] = df['message'].apply(clean_text)


In [ ]:
# token
vocab_size = 10000  
max_length = 150    
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"   

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(df['clean_message'])

sequences = tokenizer.texts_to_sequences(df['clean_message'])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [29]:
#cnn model and training
embedding_dim = 64

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    
    Conv1D(128, 5, activation='relu'),
    
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.5), 
    
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#training
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['label_num'], test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

model.save('cnn_spam_model.keras')


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.8876 - loss: 0.3230 - val_accuracy: 0.9803 - val_loss: 0.0765
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step - accuracy: 0.9874 - loss: 0.0490 - val_accuracy: 0.9848 - val_loss: 0.0514
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.9978 - loss: 0.0120 - val_accuracy: 0.9848 - val_loss: 0.0584
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9998 - loss: 0.0029 - val_accuracy: 0.9848 - val_loss: 0.0650
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.9996 - loss: 0.0014 - val_accuracy: 0.9865 - val_loss: 0.0719
